# AudioControl Lab — GPU experiment
**Maëva Noukoua**

This notebook runs the real AudioLDM2 adapter. No neural results are precomputed. Select a GPU runtime first. Model downloads require network access and several GB of disk space. Start with six clips; the full benchmark has 216 clips. Read the model license before use.


In [ ]:
import sys, subprocess
from pathlib import Path
PROJECT = Path('/content/audiocontrol-lab')
if not PROJECT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/noukoua-maeva/audiocontrol-lab.git', str(PROJECT)], check=True)
%cd /content/audiocontrol-lab
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[generation]'], check=True)


If you already imported torch, transformers or diffusers before installation, restart the runtime and resume below. The neural integration is not yet verified on GPU in the initial release. Report any dependency or memory failure rather than replacing it with synthetic results.


In [ ]:
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
print(torch.__version__, torch.cuda.get_device_name(0))


In [ ]:
from audiocontrol.experiment import generate, load_config, design
from audiocontrol.evaluate import evaluate
CONFIG = 'configs/smoke.json'
RUN = 'runs/audioldm2-smoke'
print('Clips:', len(list(design(load_config(CONFIG)))))


In [ ]:
generate(CONFIG, RUN, backend='audioldm2', device='cuda', offload=True)
evaluate(RUN)


In [ ]:
from IPython.display import Audio, display
from scipy.io import wavfile
import json
manifest = json.loads((Path(RUN) / 'manifest.json').read_text())
for record in manifest['records']:
    print(record['prompt'])
    sr, audio = wavfile.read(Path(RUN) / record['file'])
    display(Audio(audio, rate=sr, normalize=False))


Listen to every smoke clip. Check that the sound source is plausible, pitch estimation is meaningful, and event detection is reasonable. Record failures. Fix the protocol on a separate development set, then freeze it before running the full benchmark. Same seeds reduce one source of variation but do not preserve sound identity.


In [ ]:
# Run this only after the smoke test works and you have enough runtime.
RUN_FULL_BENCHMARK = False
if RUN_FULL_BENCHMARK:
    CONFIG = 'configs/benchmark.json'
    RUN = 'runs/audioldm2-benchmark'
    generate(CONFIG, RUN, backend='audioldm2', device='cuda', offload=True)
    evaluate(RUN)


In [ ]:
import shutil
archive = shutil.make_archive(str(Path(RUN)), 'zip', RUN)
print('Saved:', archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('Download the archive from your notebook file browser.')


## What to publish after running
Commit the resolved model revision, configuration, environment versions, metrics, complete failure counts and a balanced audio sample. Keep all clips in the downloadable run archive. Do not claim training, fine-tuning, novel model architecture, or improved generation quality from this inference-only experiment.
